In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np

import matplotlib
import matplotlib.animation as animation
import matplotlib.pyplot as plt
from IPython.display import HTML
import PIL.Image

from dm_control import mjcf
from dm_control import viewer

from swarmbots.unit import Unit
from swarmbots.rendering import display_video

In [2]:
RENDER_WIDTH = 640
RENDER_HEIGHT = 480

In [3]:
env = mjcf.RootElement()

getattr(env.visual, 'global').offwidth = RENDER_WIDTH
getattr(env.visual, 'global').offheight = RENDER_HEIGHT

chequered = env.asset.add('texture', type='2d', builtin='checker', width=300,
                            height=300, rgb1=[.2, .3, .4], rgb2=[.3, .4, .5])
grid = env.asset.add('material', name='grid', texture=chequered,
                       texrepeat=[5, 5], reflectance=.2)
env.worldbody.add('geom', type='plane', size=[2, 2, .1], material=grid)

for x in [-2, 2]:
  env.worldbody.add('light', pos=[x, -1, 3], dir=[-x, 1, -2])

unit = Unit(0.1, 0.2, 0.025, np.pi/2)

spawn_site = env.worldbody.add('site', pos=[0, 0, 0.5])#, zaxis=[-np.sqrt(2/9), np.sqrt(2/3), -1/3])#euler=[0, np.pi, np.pi/2])
spawn_site.attach(unit.model).add('freejoint')
# env.to_xml_string()

unit = Unit(0.1, 0.2, 0.025, np.pi/2)
# 
spawn_site = env.worldbody.add('site', pos=[0.5, 0, 0.5], euler=[0, np.pi, 0]) #, zaxis=[-np.sqrt(2/9), np.sqrt(2/3), -1/3])#euler=[0, np.pi, np.pi/2])
spawn_site.attach(unit.model).add('freejoint')

print(env.worldbody.find_all('body'))

env.equality.add('weld', body1='unnamed_model/unnamed_model/', body2='unnamed_model_1/unnamed_model/')
env.equality.add('weld', body1='unnamed_model/unnamed_model_1/', body2='unnamed_model_1/unnamed_model_1/')
env.equality.add('weld', body1='unnamed_model/unnamed_model_2/', body2='unnamed_model_1/unnamed_model_2/')
env.equality.add('weld', body1='unnamed_model/unnamed_model_3/', body2='unnamed_model_1/unnamed_model_3/')

env.to_xml_string()

[MJCF Element: <body pos="0 0 0.5" name="unnamed_model/">...</body>, MJCF Element: <body pos="0 0 0.10000000000000001" euler="0 0 0" name="unnamed_model/">...</body>, MJCF Element: <body>...</body>, MJCF Element: <body pos="0.094280904158206336 0 -0.033333333333333333" zaxis="0.094280904158206336 0 -0.033333333333333333" name="unnamed_model_1/">...</body>, MJCF Element: <body>...</body>, MJCF Element: <body pos="-0.047140452079103168 0.081649658092772609 -0.033333333333333333" zaxis="-0.047140452079103168 0.081649658092772609 -0.033333333333333333" name="unnamed_model_2/">...</body>, MJCF Element: <body>...</body>, MJCF Element: <body pos="-0.047140452079103168 -0.081649658092772609 -0.033333333333333333" zaxis="-0.047140452079103168 -0.081649658092772609 -0.033333333333333333" name="unnamed_model_3/">...</body>, MJCF Element: <body>...</body>, MJCF Element: <body pos="0.5 0 0.5" euler="0 3.1415926535897931 0" name="unnamed_model_1/">...</body>, MJCF Element: <body pos="0 0 0.100000000

'<mujoco model="unnamed_model">\n  <compiler angle="radian"/>\n  <visual>\n    <global offwidth="640" offheight="480"/>\n  </visual>\n  <default>\n    <default class="/"/>\n    <default class="unnamed_model/"/>\n    <default class="unnamed_model/unnamed_model/"/>\n    <default class="unnamed_model/unnamed_model_1/"/>\n    <default class="unnamed_model/unnamed_model_2/"/>\n    <default class="unnamed_model/unnamed_model_3/"/>\n    <default class="unnamed_model_1/"/>\n    <default class="unnamed_model_1/unnamed_model/"/>\n    <default class="unnamed_model_1/unnamed_model_1/"/>\n    <default class="unnamed_model_1/unnamed_model_2/"/>\n    <default class="unnamed_model_1/unnamed_model_3/"/>\n  </default>\n  <asset>\n    <texture name="//unnamed_texture_0" type="2d" builtin="checker" rgb1="0.20000000000000001 0.29999999999999999 0.40000000000000002" rgb2="0.29999999999999999 0.40000000000000002 0.5" width="300" height="300"/>\n    <material name="grid" class="/" texture="//unnamed_texture_0

In [4]:
physics = mjcf.Physics.from_mjcf_model(env)
PIL.Image.fromarray(physics.render(width=RENDER_WIDTH, height=RENDER_HEIGHT))

print(physics.data.eq_active)

[1 1 1 1]


In [5]:
duration = 10   # (Seconds)
framerate = 30  # (Hz)
video = []

last_switch = 0

# Simulate, saving video frames and torso locations.
physics.reset()
while physics.data.time < duration:
  physics.step()

  # Save video frames.
  if len(video) < physics.data.time * framerate:
    pixels = physics.render(width=RENDER_WIDTH, height=RENDER_HEIGHT)
    video.append(pixels.copy())
    
  if (physics.data.time - last_switch) > 0.5:
    last_switch = physics.data.time
    for i in range(4):
        physics.data.eq_active[i] = 1 - physics.data.eq_active[i]
    print(last_switch)

display_video(video, framerate)

0.5000000000000003
1.0000000000000007
1.500000000000001
2.0000000000000013
2.501999999999946
3.0039999999998908
3.5059999999998355
4.007999999999781
4.509999999999725
5.01199999999967
5.513999999999615
6.0159999999995595
6.517999999999504
7.019999999999449
7.521999999999394
8.023999999999349
8.523999999999516
9.023999999999683
9.52399999999985
